In [1]:
import pandas as pd

In [2]:
print("Hello")

Hello


In [7]:
!pip install psycopg2-binary dotenv

In [8]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

usr = os.environ.get("POSTGRES_USER")
pwd = os.environ.get("POSTGRES_PASSWORD")
dbs = os.environ.get("POSTGRES_DB")

In [10]:
import psycopg2

conn = psycopg2.connect(
    host="db",         # or "localhost" if notebook is on same machine
    database=dbs,   # change if needed
    user=usr,     # match your Docker credentials
    password=pwd, 
    port=5432          # default PostgreSQL port
)

cur = conn.cursor()

In [5]:
cur.execute("""
CREATE TABLE IF NOT EXISTS students (
    id SERIAL PRIMARY KEY,
    name VARCHAR(100),
    birth_date DATE,
    class VARCHAR(50)
);
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS course (
    id SERIAL PRIMARY KEY,
    name VARCHAR(100),
    duration INTEGER,
    level VARCHAR(50)
);
""")

conn.commit()


In [4]:
!pip install faker


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 21.7 MB/s eta 0:00:0000:0100:01


In [5]:
from faker import Faker
from random import randint, choice
import datetime

fake = Faker()

# Insert into students
for _ in range(10):
    name = fake.name()
    birth_date = fake.date_between(start_date="-20y", end_date="-10y")
    class_name = choice(['A', 'B', 'C'])
    cur.execute("INSERT INTO students (name, birth_date, class) VALUES (%s, %s, %s)",
                (name, birth_date, class_name))

# Insert into course
for _ in range(5):
    name = fake.word().capitalize()
    duration = randint(1, 12)
    level = choice(['Beginner', 'Intermediate', 'Advanced'])
    cur.execute("INSERT INTO course (name, duration, level) VALUES (%s, %s, %s)",
                (name, duration, level))

conn.commit()


In [11]:
import pandas as pd

students_df = pd.read_sql("SELECT * FROM students", conn)
courses_df = pd.read_sql("SELECT * FROM course", conn)

print("Students Table:")
display(students_df)

Students Table:


/tmp/ipykernel_132/3322251991.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  students_df = pd.read_sql("SELECT * FROM students", conn)
/tmp/ipykernel_132/3322251991.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  courses_df = pd.read_sql("SELECT * FROM course", conn)


,id,name,birth_date,class
0,1,Tammie Mcgee,2012-03-09,A
1,2,Joseph West,2013-07-22,B
2,3,Robert Sampson,2006-09-26,C
3,4,Terry Turner,2006-03-05,C
4,5,Michael Aguilar,2009-08-20,B
5,6,Kevin Simmons,2008-03-05,A
6,7,Danielle Fernandez,2007-06-23,B
7,8,Darryl Ellis,2005-08-05,A
8,9,April Alexander,2011-01-11,B
9,10,Marc Brown,2009-06-08,A


In [12]:
print("Courses Table:")
display(courses_df)

Courses Table:


,id,name,duration,level
0,1,Kitchen,1,Intermediate
1,2,Above,7,Advanced
2,3,Someone,5,Advanced
3,4,Miss,8,Advanced
4,5,Movie,10,Beginner


In [13]:
cur.close()

In [14]:
conn.close()